# Recommender Systeme

## Was ist ein Empfehlungssystem?

```
Netflix: "Weil du Inception gesehen hast..."
Amazon: "Kunden kauften auch..."
Spotify: Dein Discover Weekly
```

**Drei Haupttypen:**
1. **Content-Based Filtering** — empfiehlt ähnliche Items (nach Eigenschaften)
2. **Collaborative Filtering** — empfiehlt was ähnliche Nutzer mochten
3. **Hybrid** — Kombination aus beidem

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from scipy.sparse.linalg import svds

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

## Teil 1: Content-Based Filtering

```
Idee: Items mit ähnlichen EIGENSCHAFTEN empfehlen

Workflow:
1. Items als Feature-Vektor darstellen
2. Ähnlichkeit berechnen (Cosine Similarity)
3. Ähnlichste Items empfehlen
```

### Cosine Similarity
Misst den Winkel zwischen zwei Vektoren:
- **1** = identisch
- **0** = völlig verschieden

In [ ]:
# Demo-Datensatz: 20 bekannte Filme
movies = pd.DataFrame({
    'title': ['The Shawshank Redemption', 'The Godfather', 'The Dark Knight',
              '12 Angry Men', "Schindler's List",
              'The Lord of the Rings: The Return of the King', 'Pulp Fiction',
              'The Good, the Bad and the Ugly',
              'The Lord of the Rings: The Fellowship of the Ring', 'Fight Club',
              'Forrest Gump', 'Inception',
              'The Lord of the Rings: The Two Towers',
              "One Flew Over the Cuckoo's Nest", 'Goodfellas',
              'The Matrix', 'Seven Samurai',
              'Star Wars: Episode IV - A New Hope', 'City of God', 'Se7en'],
    'release_year': [1994, 1972, 2008, 1957, 1993, 2003, 1994, 1966, 2001,
                     1999, 1994, 2010, 2002, 1975, 1990, 1999, 1954, 1977, 2002, 1995],
    'genre': ['Drama', 'Crime', 'Action', 'Drama', 'Biography', 'Adventure',
              'Crime', 'Western', 'Adventure', 'Drama', 'Comedy', 'Action',
              'Adventure', 'Drama', 'Crime', 'Sci-Fi', 'Action', 'Adventure',
              'Crime', 'Crime'],
    'director': ['Frank Darabont', 'Francis Ford Coppola', 'Christopher Nolan',
                 'Sidney Lumet', 'Steven Spielberg', 'Peter Jackson',
                 'Quentin Tarantino', 'Sergio Leone', 'Peter Jackson',
                 'David Fincher', 'Robert Zemeckis', 'Christopher Nolan',
                 'Peter Jackson', 'Milos Forman', 'Martin Scorsese',
                 'Lana Wachowski', 'Akira Kurosawa', 'George Lucas',
                 'Fernando Meirelles', 'David Fincher'],
    'actor1': ['Tim Robbins', 'Marlon Brando', 'Christian Bale', 'Henry Fonda',
               'Liam Neeson', 'Elijah Wood', 'John Travolta', 'Clint Eastwood',
               'Elijah Wood', 'Brad Pitt', 'Tom Hanks', 'Leonardo DiCaprio',
               'Elijah Wood', 'Jack Nicholson', 'Robert De Niro',
               'Keanu Reeves', 'Toshirô Mifune', 'Mark Hamill',
               'Alexandre Rodrigues', 'Brad Pitt'],
    'actor2': ['Morgan Freeman', 'Al Pacino', 'Heath Ledger', 'Lee J. Cobb',
               'Ralph Fiennes', 'Viggo Mortensen', 'Uma Thurman', 'Eli Wallach',
               'Ian McKellen', 'Edward Norton', 'Robin Wright',
               'Joseph Gordon-Levitt', 'Viggo Mortensen', 'Louise Fletcher',
               'Ray Liotta', 'Laurence Fishburne', 'Takashi Shimura',
               'Harrison Ford', 'Leandro Firmino', 'Morgan Freeman']
})

print(f"Filme im Datensatz: {len(movies)}")
movies.head(3)

In [ ]:
# Features vorbereiten: kategorische Spalten → One-Hot Encoding
features = pd.get_dummies(movies[['release_year', 'genre', 'director', 'actor1', 'actor2']],
                          drop_first=True)

# Jahreszahl skalieren (sonst dominiert sie die Ähnlichkeit)
scaler = StandardScaler()
features['release_year'] = scaler.fit_transform(features[['release_year']])

print(f"Feature-Matrix: {features.shape}")
features.head(2)

In [ ]:
# Cosine Similarity Matrix berechnen
cosine_sim = cosine_similarity(features, features)

print(f"Ähnlichkeitsmatrix: {cosine_sim.shape}")
print(f"(Jede Zeile/Spalte = ein Film, Wert = Ähnlichkeit)")

# Als DataFrame für bessere Lesbarkeit
sim_df = pd.DataFrame(cosine_sim, index=movies['title'], columns=movies['title'])
sim_df.iloc[:3, :3].round(3)

In [ ]:
# Empfehlungsfunktion
def recommend_content(title, n=3):
    idx = movies.index[movies['title'] == title][0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top = scores[1:n+1]   # erstes überspringen = Film selbst
    indices = [i[0] for i in top]
    return movies[['title', 'genre', 'director']].iloc[indices]

print("Empfehlungen für 'The Matrix':")
recommend_content('The Matrix')

In [ ]:
# Visualisierung: Ähnlichkeits-Heatmap (erste 8 Filme)
plt.figure(figsize=(10, 8))
sns.heatmap(cosine_sim[:8, :8],
            xticklabels=movies['title'][:8],
            yticklabels=movies['title'][:8],
            annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Cosine Similarity zwischen Filmen')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Teil 2: Collaborative Filtering

```
Idee: Nutzer mit ähnlichem Geschmack haben dieselben Vorlieben

User-Item-Matrix:
             Film1  Film2  Film3  Film4
  Nutzer1:     5      3      ?      1    ← Was würde Nutzer1 Film3 geben?
  Nutzer2:     4      0      4      1
  Nutzer3:     1      1      0      5

Problem: Sehr sparse — die meisten Felder sind leer!
Lösung: SVD findet latente Faktoren (versteckte Muster)
```

In [ ]:
# Demo-Daten: User-Item-Ratings
ratings_data = {
    'userId':  [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5],
    'title':   ['The Matrix', 'Inception', 'The Dark Knight',
                'The Matrix', 'Fight Club', 'Se7en',
                'Forrest Gump', 'The Shawshank Redemption', '12 Angry Men',
                'Inception', 'The Dark Knight', 'Fight Club',
                'Forrest Gump', '12 Angry Men', 'The Godfather'],
    'rating':  [5, 4, 5, 5, 4, 3, 4, 5, 4, 3, 4, 5, 5, 4, 5]
}
ratings_df = pd.DataFrame(ratings_data)

# User-Item-Matrix erstellen
user_item = ratings_df.pivot_table(index='userId', columns='title',
                                    values='rating', fill_value=0)
print(f"User-Item-Matrix: {user_item.shape}")
user_item

In [ ]:
# Rating-Zentrierung: Nutzer-Bias entfernen
# Manche Nutzer geben generell hohe/niedrige Noten → normalisieren!
ratings_df['mean_rating']     = ratings_df.groupby('userId')['rating'].transform('mean')
ratings_df['centered_rating'] = ratings_df['rating'] - ratings_df['mean_rating']

print("Vor/nach Zentrierung (Nutzer 1):")
ratings_df[ratings_df['userId'] == 1][['title', 'rating', 'mean_rating', 'centered_rating']]

In [ ]:
# SVD: Matrix in latente Faktoren zerlegen
user_item_centered = ratings_df.pivot_table(index='userId', columns='title',
                                             values='centered_rating', fill_value=0)
ratings_matrix = np.array(user_item_centered)

# k = Anzahl latenter Faktoren (Kompromiss: zu viele = Overfitting)
k = 2
U, sigma, Vt = svds(ratings_matrix, k=k)
sigma = np.diag(sigma)

print(f"U (Nutzer-Faktoren):   {U.shape}")
print(f"sigma (Gewichtungen):  {sigma.shape}")
print(f"Vt (Item-Faktoren):    {Vt.shape}")

In [ ]:
# Ähnlichkeit zwischen Nutzern und Items berechnen
items_df = pd.DataFrame(Vt, columns=user_item_centered.columns).T
user_df  = pd.DataFrame(U)

user_similarity = cosine_similarity(user_df, user_df)
item_similarity = cosine_similarity(items_df, items_df)

user_sim_df = pd.DataFrame(user_similarity,
                            index=user_item_centered.index,
                            columns=user_item_centered.index)
print("Nutzer-Ähnlichkeitsmatrix:")
user_sim_df.round(3)

In [ ]:
# Empfehlungsfunktion für Items
item_sim_df = pd.DataFrame(item_similarity,
                            index=items_df.index,
                            columns=items_df.index)

def recommend_collab(title, n=3):
    similar = item_sim_df[title].sort_values(ascending=False)
    return similar.iloc[1:n+1]   # erstes = der Film selbst

print("Ähnliche Filme zu 'Inception' (Collaborative Filtering):")
recommend_collab('Inception')

## Teil 3: Surprise Library — ML für Ratings

Die **Surprise**-Bibliothek macht Rating-Vorhersagen einfach:
- SVD-Algorithmus eingebaut
- Cross-Validation
- RMSE als Fehlermaß

In [ ]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

# Surprise braucht: userId, itemId, rating
ratings_for_surprise = ratings_df[['userId', 'title', 'rating']].copy()
ratings_for_surprise.columns = ['userId', 'itemId', 'rating']

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_for_surprise, reader)

trainset, testset = train_test_split(data, test_size=0.25, random_state=42)

model = SVD(random_state=42)
model.fit(trainset)

predictions = model.test(testset)
print(f"RMSE: {accuracy.rmse(predictions):.4f}")
print("(RMSE = durchschnittlicher Vorhersage-Fehler in Rating-Punkten)")

In [ ]:
# Einzelne Vorhersage
user_id = 1
item_id = 'Forrest Gump'
pred = model.predict(uid=user_id, iid=item_id)

print(f"Vorhergesagtes Rating von Nutzer {user_id} für '{item_id}': {pred.est:.2f}")

## Vergleich: Content-Based vs. Collaborative

| | Content-Based | Collaborative |
|--|:---:|:---:|
| **Braucht Item-Infos** | ✅ Ja | ❌ Nein |
| **Braucht User-History** | ❌ Nein | ✅ Ja |
| **Kaltstart (neuer User)** | ✅ kein Problem | ❌ Problem |
| **Überraschende Empfehlungen** | ❌ selten | ✅ möglich |
| **Filter Bubble** | ❌ Risiko | ✅ geringer |

**Fazit:** In der Praxis werden beide kombiniert → **Hybrid System**